In [1]:
pip install transformers torch accelerate

  Using cached transformers-5.18.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.14.1-cp311-cp311-win_amd64.whl.metadata (38 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached regex-2026.9.29-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
  Using cached huggingface_hub-1.33.0-py3

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [8]:
import os
import re
import sqlite3
import torch
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

app = FastAPI(title="Text-to-SQL Warehouse API", version="1.0")

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & MÔ HÌNH TRÊN Ổ D
# ==========================================
DB_DIR = r"D:\đại học quy nhơn\thực hành chuyên đề\phat_trien_he_thong_ql_kho"
DB_PATH = os.path.join(DB_DIR, "quan_ly_kho.db")

CACHE_DIR = r"D:\huggingface_models"
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print("Đang tải mô hình Text-to-SQL từ ổ D...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, 
    cache_dir=CACHE_DIR
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16, 
    device_map="auto"
)
print("Tải mô hình thành công!")

# ==========================================
# SCHEMA CƠ SỞ DỮ LIỆU (9 BẢNG)
# ==========================================
database_schema = """
CREATE TABLE NhanVien (
    MaNV TEXT PRIMARY KEY,
    HoTen TEXT,
    ChucVu TEXT
);

CREATE TABLE NhaCungCap (
    MaNCC TEXT PRIMARY KEY,
    TenNCC TEXT,
    DiaChi TEXT,
    DienThoai TEXT
);

CREATE TABLE VatTu (
    MaVT TEXT PRIMARY KEY,
    TenVT TEXT,
    DonViTinh TEXT,
    GiaGoc REAL
);

CREATE TABLE Kho (
    MaKho TEXT PRIMARY KEY,
    TenKho TEXT,
    DiaChi TEXT
);

CREATE TABLE TonKho (
    MaKho TEXT,
    MaVT TEXT,
    SoLuongTon INTEGER,
    PRIMARY KEY (MaKho, MaVT)
);

CREATE TABLE PhieuNhap (
    MaPN TEXT PRIMARY KEY,
    NgayNhap TEXT,
    MaNCC TEXT,
    MaNV TEXT
);

CREATE TABLE ChiTietPhieuNhap (
    MaPN TEXT,
    MaVT TEXT,
    SoLuong INTEGER,
    DonGia REAL,
    PRIMARY KEY (MaPN, MaVT)
);

CREATE TABLE PhieuXuat (
    MaPX TEXT PRIMARY KEY,
    NgayXuat TEXT,
    NguoiNhan TEXT,
    MaNV TEXT
);

CREATE TABLE ChiTietPhieuXuat (
    MaPX TEXT,
    MaVT TEXT,
    SoLuong INTEGER,
    DonGia REAL,
    PRIMARY KEY (MaPX, MaVT)
);
"""

# ==========================================
# HÀM XỬ LÝ TEXT-TO-SQL & THỰC THI
# ==========================================
def generate_and_execute_sql(user_query: str):
    # 1. Xây dựng prompt chuẩn ChatML của Qwen
    prompt = f"""<|im_start|>system
Bạn là một công cụ dịch Text-to-SQL cho SQLite chuyên nghiệp. Chỉ xuất ra câu lệnh SQL hợp lệ dựa vào schema sau:
{database_schema}
<|im_end|>
<|im_start|>user
{user_query}<|im_end|>
<|im_start|>assistant
SELECT """

    # 2. Tokenize và sinh câu lệnh từ model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=150, 
        temperature=0.01, 
        do_sample=False
    )
    
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 3. Ghép lại chữ SELECT và dùng Regex lấy trọn vẹn câu lệnh SQL
    full_text = "SELECT " + raw_output.split("SELECT")[-1] if "SELECT" in raw_output else raw_output
    
    match = re.search(r"(SELECT\s+.*?;)", full_text, re.IGNORECASE | re.DOTALL)
    if match:
        sql_query = match.group(1)
    else:
        sql_query = full_text.split("\n")[0].strip()

    # 4. Thực thi câu lệnh SQL vào SQLite
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()

        return {
            "status": "success",
            "sql_query": sql_query,
            "data": rows
        }
    except Exception as e:
        return {
            "status": "error",
            "sql_query": sql_query,
            "message": str(e)
        }

# ==========================================
# API ENDPOINT CHO FASTAPI
# ==========================================
@app.get("/api/chat")
def text_to_sql_endpoint(prompt: str):
    prompt_lower = prompt.lower().strip()
    
    if any(w in prompt_lower for w in ["chào", "hi", "hello", "bạn là ai"]):
        return {"status": "success", "reply": "👋 Chào bạn, tôi là trợ lý AI quản lý kho VLXD."}

    # Gọi trực tiếp hàm xử lý Text-to-SQL đã viết phía trên
    result = generate_and_execute_sql(prompt)

    if result["status"] == "error":
        return {
            "status": "error", 
            "reply": f"⚠️ Lỗi thực thi SQL: {result['message']} \n- Câu lệnh gây lỗi: `{result['sql_query']}`"
        }

    rows = result["data"]
    sql_query = result["sql_query"]

    if not rows:
        return {"status": "success", "reply": f"📊 Câu lệnh SQL sinh ra: `{sql_query}`\n\nKhông tìm thấy dữ liệu phù hợp trong cơ sở dữ liệu."}
        
    result_str = f"🔍 **SQL:** `{sql_query}`\n\n📊 **Kết quả:**\n"
    for row in rows:
        result_str += f"- {row}\n"
        
    return {"status": "success", "reply": result_str}

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'Qwen/Qwen2.5-Coder-1.5B-Instruct'.


Đang tải mô hình Text-to-SQL từ ổ D...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:16<00:00, 20.39it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Tải mô hình thành công!


In [7]:
generate_and_execute_sql("Liệt kê toàn bộ danh sách nhân viên trong kho")

{'status': 'error',
 'sql_query': 'SELECT   NV.HoTen FROM NhanVien AS NV INNER JOIN Kho AS K ON NV.MaNV = K.MaNV',
 'message': 'no such table: NhanVien'}

In [1]:
!pip install datasets -q
print("Cài đặt thành công thư viện datasets!")

Cài đặt thành công thư viện datasets!


In [5]:
from datasets import load_dataset
import pandas as pd

# Tải trực tiếp bộ dữ liệu thông qua tên repository trên Hugging Face
print("Đang tải dataset xlangai/spider...")
spider_data = load_dataset("xlangai/spider")

# Kiểm tra cấu trúc dữ liệu
print(spider_data)

# Chuyển đổi tập 'train' sang DataFrame để hiển thị
df_train = pd.DataFrame(spider_data['train'])
print("\n--- 5 mẫu dữ liệu đầu tiên ---")
display(df_train[['question', 'query']].head(5))

Đang tải dataset xlangai/spider...


C:\Users\vietr\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vietr\.cache\huggingface\hub\datasets--xlangai--spider. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\vietr\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_dow

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})

--- 5 mẫu dữ liệu đầu tiên ---


,question,query
0,How many heads of the departments are older th...,SELECT count(*) FROM head WHERE age > 56
1,"List the name, born state and age of the heads...","SELECT name , born_state , age FROM head ORD..."
2,"List the creation year, name and budget of eac...","SELECT creation , name , budget_in_billions ..."
3,What are the maximum and minimum budget of the...,"SELECT max(budget_in_billions) , min(budget_i..."
4,What is the average number of employees of the...,SELECT avg(num_employees) FROM department WHER...


In [6]:
# Lấy ngẫu nhiên 3 mẫu từ tập train của Spider để test
sample_df = df_train.sample(3)

for idx, row in sample_df.iterrows():
    print(f"🔹 Cơ sở dữ liệu (db_id): {row['db_id']}")
    print(f"❓ Câu hỏi gốc: {row['question']}")
    print(f"🎯 SQL thực tế (Ground-truth): {row['query']}")
    print("-" * 60)

🔹 Cơ sở dữ liệu (db_id): college_1
❓ Câu hỏi gốc: Find the number of classes in each school.
🎯 SQL thực tế (Ground-truth): SELECT count(*) ,  T3.school_code FROM CLASS AS T1 JOIN course AS T2 ON T1.crs_code  =  T2.crs_code JOIN department AS T3 ON T2.dept_code  =  T3.dept_code GROUP BY T3.school_code
------------------------------------------------------------
🔹 Cơ sở dữ liệu (db_id): tracking_grants_for_research
❓ Câu hỏi gốc: What are the descriptions of all the project outcomes?
🎯 SQL thực tế (Ground-truth): SELECT T1.outcome_description FROM Research_outcomes AS T1 JOIN Project_outcomes AS T2 ON T1.outcome_code  =  T2.outcome_code
------------------------------------------------------------
🔹 Cơ sở dữ liệu (db_id): cre_Doc_Control_Systems
❓ Câu hỏi gốc: What is the receipt date of the document with id 3?
🎯 SQL thực tế (Ground-truth): SELECT receipt_date FROM Documents WHERE document_id = 3;
------------------------------------------------------------


In [7]:
# gọi lại dường dẫn dataset
load_dataset("xlangai/spider")

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})

In [ ]:
# Lấy ngẫu nhiên 1 mẫu từ tập dữ liệu Spider (`df_train`)
sample_row = df_train.sample(1).iloc[0]

db_id = sample_row['db_id']
question = sample_row['question'] # Câu hỏi tiếng Anh từ Spider
true_sql = sample_row['query']     # Câu lệnh SQL chuẩn (ground-truth)

print(f"📂 Database ID: {db_id}")
print(f"❓ Câu hỏi từ Spider: {question}")
print(f"🎯 SQL chuẩn (Ground-truth): {true_sql}")

# Gọi hàm sinh SQL và thực thi dựa trên câu hỏi của Spider
# (Lưu ý: Do câu hỏi Spider dùng các database mẫu của họ, ta chỉ test khả năng sinh SQL của mô hình)
prompt_spider = f"Write an SQLite query for the database '{db_id}' to answer this question: {question}"
result = generate_and_execute_sql(prompt_spider)

print("-" * 50)
print(f"🤖 SQL do mô hình sinh ra:\n{result['sql_query']}")
print(f"📊 Kết quả thực thi:\n{result.get('data', result.get('message'))}")

📂 Database ID: college_1
❓ Câu hỏi từ Spider: Find the first names and degree of all professors who are teaching some class in Computer Info. Systems department.
🎯 SQL chuẩn (Ground-truth): SELECT DISTINCT T2.emp_fname ,  T3.prof_high_degree FROM CLASS AS T1 JOIN employee AS T2 ON T1.prof_num  =  T2.emp_num JOIN professor AS T3 ON T2.emp_num  =  T3.emp_num JOIN department AS T4 ON T4.dept_code  =  T3.dept_code WHERE T4.dept_name  =  'Computer Info. Systems'


In [2]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Nạp lại dataset Spider từ bộ nhớ cache
print("Đang nạp dataset Spider...")
dataset = load_dataset("xlangai/spider")
df_train = dataset['train'].to_pandas()
df_validation = dataset['validation'].to_pandas()

# Nạp lại mô hình Qwen từ ổ D
CACHE_DIR = r"D:\huggingface_models"
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print("Đang nạp mô hình Qwen từ ổ D...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16, 
    device_map="auto"
)

# Hàm sinh SQL
def generate_and_execute_sql(prompt):
    full_prompt = f"<|im_start|>system\nBạn là một chuyên gia Text-to-SQL. Chỉ xuất ra câu lệnh SQL.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\nSELECT "
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.01, do_sample=False)
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql_query = "SELECT " + raw_output.split("SELECT ")[-1] if "SELECT " in raw_output else raw_output
    return {"sql_query": sql_query.strip()}

print("Đã sẵn sàng!")

C:\Users\vietr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang nạp dataset Spider...


Đang nạp mô hình Qwen từ ổ D...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:16<00:00, 20.13it/s]
Some parameters are on the meta device because they were offloaded to the disk.


Đã sẵn sàng!


In [3]:
import sqlite3

def evaluate_sql_execution(db_path, pred_sql, true_sql):
    """
    So sánh kết quả thực thi của SQL dự đoán và SQL chuẩn trên database SQLite.
    Trả về True nếu kết quả giống nhau, False nếu khác hoặc lỗi.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    try:
        # Thực thi SQL chuẩn (Ground-truth)
        cursor.execute(true_sql)
        true_result = cursor.fetchall()
    except Exception as e:
        conn.close()
        return {"match": False, "error": f"Lỗi SQL chuẩn: {str(e)}"}
        
    try:
        # Thực thi SQL do mô hình sinh ra
        cursor.execute(pred_sql)
        pred_result = cursor.fetchall()
    except Exception as e:
        conn.close()
        return {"match": False, "error": f"Lỗi SQL mô hình: {str(e)}"}
        
    conn.close()
    
    # So sánh tập kết quả (có thể sắp xếp lại hoặc so sánh trực tiếp tùy bài toán)
    is_match = (set(true_result) == set(pred_result))
    return {"match": is_match, "true_result": true_result, "pred_result": pred_result}

In [5]:
FEW_SHOT_EXAMPLES = """
Dưới đây là một số ví dụ mẫu:

Câu hỏi: Liệt kê tên tất cả vật tư có đơn vị tính là 'Kg'
SQL: SELECT TenVT FROM VatTu WHERE DonViTinh = 'Kg';

Câu hỏi: Cho biết tổng số lượng tồn kho của vật tư có mã 'VT01'
SQL: SELECT SoLuongTon FROM TonKho WHERE MaVT = 'VT01';

Câu hỏi: Tìm họ tên nhân viên đã lập phiếu nhập số 'PN01'
SQL: SELECT HoTen FROM NhanVien JOIN PhieuNhap ON NhanVien.MaNV = PhieuNhap.MaNV WHERE PhieuNhap.MaPN = 'PN01';
"""

In [ ]:
#Tự động tạo tập dữ liệu huấn luyện (Dataset Generation)

In [4]:
import pandas as pd
import json

# Thay thế đường dẫn này bằng đường dẫn thực tế đến file trên máy tính của bạn
file_path = r"D:\đại học quy nhơn\thực hành chuyên dề\phat_trien_he_thong_ql_kho\csv\danh_sach_dan_toc_viet_nam.csv"

df = pd.read_csv(file_path)
df.columns = ['STT', 'TenDanToc', 'DanSo2019', 'TyLe']

train_data = []

for index, row in df.iterrows():
    ten_dt = row['TenDanToc']
    dan_so = row['DanSo2019']
    ty_le = row['TyLe']
    
    train_data.append({
        "instruction": "Bạn là chuyên gia Text-to-SQL. Dựa vào bảng DanhSachDanToc (STT, TenDanToc, DanSo2019, TyLe), hãy viết câu lệnh SQLite.",
        "input": f"Dân số của dân tộc {ten_dt} năm 2019 là bao nhiêu?",
        "output": f"SELECT DanSo2019 FROM DanhSachDanToc WHERE TenDanToc = '{ten_dt}';"
    })
    
    train_data.append({
        "instruction": "Bạn là chuyên gia Text-to-SQL. Dựa vào bảng DanhSachDanToc (STT, TenDanToc, DanSo2019, TyLe), hãy viết câu lệnh SQLite.",
        "input": f"Tỷ lệ dân số của dân tộc {ten_dt} chiếm bao nhiêu phần trăm?",
        "output": f"SELECT TyLe FROM DanhSachDanToc WHERE TenDanToc = '{ten_dt}';"
    })

general_queries = [
    {
        "instruction": "Bạn là chuyên gia Text-to-SQL. Dựa vào bảng DanhSachDanToc (STT, TenDanToc, DanSo2019, TyLe), hãy viết câu lệnh SQLite.",
        "input": "Top 5 dân tộc đông dân nhất Việt Nam",
        "output": "SELECT TenDanToc, DanSo2019 FROM DanhSachDanToc ORDER BY DanSo2019 DESC LIMIT 5;"
    },
    {
        "instruction": "Bạn là chuyên gia Text-to-SQL. Dựa vào bảng DanhSachDanToc (STT, TenDanToc, DanSo2019, TyLe), hãy viết câu lệnh SQLite.",
        "input": "Tổng dân số của tất cả các dân tộc là bao nhiêu?",
        "output": "SELECT SUM(DanSo2019) AS TongDanSo FROM DanhSachDanToc;"
    },
    {
        "instruction": "Bạn là chuyên gia Text-to-SQL. Dựa vào bảng DanhSachDanToc (STT, TenDanToc, DanSo2019, TyLe), hãy viết câu lệnh SQLite.",
        "input": "Liệt kê các dân tộc có tỷ lệ nhỏ hơn 0.1%",
        "output": "SELECT TenDanToc, TyLe FROM DanhSachDanToc WHERE TyLe < 0.1;"
    }
]

train_data.extend(general_queries)

with open("train_dantoc.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

print(f"Đã tạo thành công file huấn luyện với {len(train_data)} mẫu câu!")

Đã tạo thành công file huấn luyện với 111 mẫu câu!


In [6]:
pip install transformers datasets peft trl torch accelerate bitsandbytes

  Using cached bitsandbytes-0.50.2-py3-none-win_amd64.whl.metadata (11 kB)
   ---------------------------------------- 0.0/835.4 kB ? eta -:--:--
   ------------ --------------------------- 262.1/835.4 kB ? eta -:--:--
   ---------------------------------------- 835.4/835.4 kB 3.7 MB/s  0:00:00
   ---------------------------------------- 0.0/948.2 kB ? eta -:--:--
   ---------------------------------------- 948.2/948.2 kB 4.9 MB/s  0:00:00
Using cached bitsandbytes-0.50.2-py3-none-win_amd64.whl (39.1 MB)

   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 [bitsandbytes]
   ---------------------------------------- 0/3 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\vietr\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [11]:
!pip install --upgrade peft trl


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\vietr\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.14.1
Uninstalling torch-2.14.1:
  Successfully uninstalled torch-2.14.1
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN Ổ D
# ==========================================
DRIVE_D_PATH = r"D:\huggingface_models"
os.makedirs(DRIVE_D_PATH, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print(f"Đang tải tokenizer và mô hình trên CPU tại: {DRIVE_D_PATH}")

# Tải Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=DRIVE_D_PATH)
tokenizer.pad_token = tokenizer.eos_token

# Nạp mô hình chạy trên CPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=DRIVE_D_PATH,
    torch_dtype=torch.float32,
    device_map=None
)

# ==========================================
# 2. CẤU HÌNH LORA (PEFT)
# ==========================================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# ==========================================
# 3. NẠP TẬP DỮ LIỆU HUẤN LUYỆN
# ==========================================
dataset = load_dataset("json", data_files="train_dantoc.json")

def formatting_prompts_func(example):
    text = f"<|im_start|>system\n{example['instruction']}<|im_end|>\n<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"
    return text

# ==========================================
# 4. THAM SỐ HUẤN LUYỆN (Đã bỏ no_cuda)
# ==========================================
training_args = TrainingArguments(
    output_dir=r"D:\qwen_dantoc_finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=3,
    fp16=False,
    optim="adamw_torch",
)

# ==========================================
# 5. KHỞI TẠO TRAINER VÀ BẮT ĐẦU TRAIN
# ==========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
    formatting_func=formatting_prompts_func,
)

print("🚀 Bắt đầu quá trình Fine-tuning mô hình trên CPU...")
trainer.train()

# Lưu mô hình cuối cùng ra ổ D
trainer.model.save_pretrained(r"D:\qwen_dantoc_final_model")
tokenizer.save_pretrained(r"D:\qwen_dantoc_final_model")
print("🎉 Đã hoàn tất Fine-tuning và lưu toàn bộ mô hình tại D:\\qwen_dantoc_final_model!")

Đang tải tokenizer và mô hình trên CPU tại: D:\huggingface_models


Loading weights: 100%|██████████| 338/338 [00:34<00:00,  9.76it/s]
Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'Qwen/Qwen2.5-Coder-1.5B-Instruct'.


🚀 Bắt đầu quá trình Fine-tuning mô hình trên CPU...


C:\Users\vietr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.


Step,Training Loss
